# Healix Task 3 - English to Arabic Seq2Seq Translator

This notebook trains a custom Encoder-Decoder LSTM translator on Opus100 English-Arabic data, then translates Malak's `generated_captions.json` into Arabic for the next Healix stage.

Expected input from Task 2:

`generated_captions.json` with this format:

```json
{
  "image_name.jpg": "english caption"
}
```

Final output:

`translated_captions_ar.json` with the same image names and Arabic captions.

In [2]:
# Colab setup. Run this cell in Google Colab.
!pip -q install datasets huggingface_hub sacrebleu


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Optional: use this only if Hugging Face asks for a token.
# Do not write your token in the notebook before sharing it.
# from huggingface_hub import login
# login()

In [4]:
import json
import re
import string
from pathlib import Path

import numpy as np
import tensorflow as tf
from datasets import load_dataset
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

c:\Users\polaa\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TensorFlow: 2.20.0
GPU: []


## Configuration

In [5]:
# Change these paths depending on where you run the notebook.
# In Colab, upload generated_captions.json to /content/drive/MyDrive/Healix/

DRIVE_DIR = Path('/content/drive/MyDrive/Healix')
INPUT_CAPTIONS_JSON = DRIVE_DIR / 'generated_captions.json'
OUTPUT_DIR = DRIVE_DIR / 'outputs' / 'translator'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training size can be increased for better quality if GPU time is available.
MAX_TRAIN_SAMPLES = 60000
MAX_VAL_SAMPLES = 3000

MAX_SOURCE_LEN = 25
MAX_TARGET_LEN = 30
SOURCE_VOCAB_SIZE = 12000
TARGET_VOCAB_SIZE = 16000
EMBEDDING_DIM = 256
LATENT_DIM = 512
BATCH_SIZE = 128
EPOCHS = 20

START_TOKEN = 'startseq'
END_TOKEN = 'endseq'

print('Input captions:', INPUT_CAPTIONS_JSON)
print('Output dir:', OUTPUT_DIR)

Input captions: \content\drive\MyDrive\Healix\generated_captions.json
Output dir: \content\drive\MyDrive\Healix\outputs\translator


## Load Opus100 English-Arabic

In [6]:
opus = load_dataset('Helsinki-NLP/opus-100', 'ar-en')
opus

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})

In [7]:
def get_pair(example):
    item = example['translation']
    return item['en'], item['ar']

def clean_en(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_ar(text):
    text = str(text).strip()
    text = re.sub(r'[\u064b-\u065f\u0670]', '', text)  # Arabic diacritics
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_pairs(split_name, limit):
    src, dec_in, dec_out = [], [], []
    for ex in opus[split_name]:
        en, ar = get_pair(ex)
        en = clean_en(en)
        ar = clean_ar(ar)
        if not en or not ar:
            continue
        if len(en.split()) > MAX_SOURCE_LEN or len(ar.split()) > MAX_TARGET_LEN - 1:
            continue
        src.append(en)
        dec_in.append(f'{START_TOKEN} {ar}')
        dec_out.append(f'{ar} {END_TOKEN}')
        if len(src) >= limit:
            break
    return np.array(src), np.array(dec_in), np.array(dec_out)

train_src, train_dec_in, train_dec_out = build_pairs('train', MAX_TRAIN_SAMPLES)

# Opus100 sometimes has validation/test splits depending on the config.
val_split = 'validation' if 'validation' in opus else 'test'
val_src, val_dec_in, val_dec_out = build_pairs(val_split, MAX_VAL_SAMPLES)

print('Train pairs:', len(train_src))
print('Val pairs:', len(val_src))
print('Sample EN:', train_src[0])
print('Sample AR:', train_dec_out[0])

Train pairs: 60000
Val pairs: 1827
Sample EN: and this
Sample AR: و هذه؟ endseq


## Vectorization

In [8]:
source_vectorizer = layers.TextVectorization(
    max_tokens=SOURCE_VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SOURCE_LEN,
    standardize=None,
)

target_vectorizer = layers.TextVectorization(
    max_tokens=TARGET_VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_TARGET_LEN,
    standardize=None,
)

source_vectorizer.adapt(train_src)
target_vectorizer.adapt(np.concatenate([train_dec_in, train_dec_out]))

source_vocab = source_vectorizer.get_vocabulary()
target_vocab = target_vectorizer.get_vocabulary()
target_index_lookup = dict(enumerate(target_vocab))
target_word_to_index = {word: idx for idx, word in enumerate(target_vocab)}

print('Source vocab:', len(source_vocab))
print('Target vocab:', len(target_vocab))
print('startseq id:', target_vocab.index(START_TOKEN))
print('endseq id:', target_vocab.index(END_TOKEN))
print('[UNK] id:', target_word_to_index.get('[UNK]'))

Source vocab: 12000
Target vocab: 16000
startseq id: 2
endseq id: 3
[UNK] id: 1


In [9]:
train_encoder_input = source_vectorizer(train_src).numpy()
train_decoder_input = target_vectorizer(train_dec_in).numpy()
train_decoder_output = target_vectorizer(train_dec_out).numpy()

val_encoder_input = source_vectorizer(val_src).numpy()
val_decoder_input = target_vectorizer(val_dec_in).numpy()
val_decoder_output = target_vectorizer(val_dec_out).numpy()

print('Training arrays ready')
print('encoder input:', train_encoder_input.shape)
print('decoder input:', train_decoder_input.shape)
print('decoder output:', train_decoder_output.shape)

Training arrays ready
encoder input: (60000, 25)
decoder input: (60000, 30)
decoder output: (60000, 30)


## Build Seq2Seq LSTM Model

In [10]:
encoder_inputs = layers.Input(shape=(MAX_SOURCE_LEN,), name='encoder_input')
encoder_emb = layers.Embedding(len(source_vocab), EMBEDDING_DIM, mask_zero=False, name='encoder_embedding')(encoder_inputs)
_, state_h, state_c = layers.LSTM(LATENT_DIM, return_state=True, name='encoder_lstm')(encoder_emb)

decoder_inputs = layers.Input(shape=(MAX_TARGET_LEN,), name='decoder_input')
decoder_emb = layers.Embedding(len(target_vocab), EMBEDDING_DIM, mask_zero=False, name='decoder_embedding')(decoder_inputs)
decoder_outputs = layers.LSTM(LATENT_DIM, return_sequences=True, name='decoder_lstm')(decoder_emb, initial_state=[state_h, state_c])
decoder_outputs = layers.Dropout(0.3)(decoder_outputs)
decoder_outputs = layers.Dense(len(target_vocab), activation='softmax', name='decoder_output')(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

loss_obj = tf.keras.losses.SparseCategoricalCrossentropy(reduction='none')

def masked_loss(y_true, y_pred):
    loss = loss_obj(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, 0), loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def masked_accuracy(y_true, y_pred):
    y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    matches = tf.cast(tf.equal(y_true, y_pred_ids), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    return tf.reduce_sum(matches * mask) / tf.reduce_sum(mask)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=masked_loss,
    metrics=[masked_accuracy],
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 25, 256)   │  3,072,000 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 30, 256)   │  4,096,000 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 512),     │  1,574,912 │ encoder_embeddin… │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ (None, 30, 512)   │  1,574,912 │ decoder_embeddin… │
│                     │                   │            │ encoder_lstm[0][… │
│                     │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 30, 512)   │          0 │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 30, 16000) │  8,208,000 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 18,525,824 (70.67 MB)

 Trainable params: 18,525,824 (70.67 MB)

 Non-trainable params: 0 (0.00 B)

## Train

In [11]:
checkpoint_path = OUTPUT_DIR / 'best_en_ar_seq2seq.keras'

callbacks = [
    ModelCheckpoint(str(checkpoint_path), monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

history = model.fit(
    [train_encoder_input, train_decoder_input],
    train_decoder_output,
    validation_data=([val_encoder_input, val_decoder_input], val_decoder_output),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(OUTPUT_DIR / 'en_ar_seq2seq_final.keras')

with open(OUTPUT_DIR / 'source_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(source_vocab, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / 'target_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(target_vocab, f, ensure_ascii=False, indent=2)

print('Saved model and vocab files to:', OUTPUT_DIR)

Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 6.5133 - masked_accuracy: 0.2095
Epoch 1: val_loss improved from None to 5.45464, saving model to \content\drive\MyDrive\Healix\outputs\translator\best_en_ar_seq2seq.keras
469/469 ━━━━━━━━━━━━━━━━━━━━ 746s 1s/step - loss: 6.0846 - masked_accuracy: 0.2432 - val_loss: 5.4546 - val_masked_accuracy: 0.3035 - learning_rate: 0.0010
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 5.5977 - masked_accuracy: 0.2870
Epoch 2: val_loss improved from 5.45464 to 5.23036, saving model to \content\drive\MyDrive\Healix\outputs\translator\best_en_ar_seq2seq.keras
469/469 ━━━━━━━━━━━━━━━━━━━━ 530s 1s/step - loss: 5.5600 - masked_accuracy: 0.2886 - val_loss: 5.2304 - val_masked_accuracy: 0.3176 - learning_rate: 0.0010
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 5.3152 - masked_accuracy: 0.2990
Epoch 3: val_loss improved from 5.23036 to 5.09790, saving model to \content\drive\MyDrive\Healix\outputs\translator\best_en_ar_seq

## Greedy Translation

In [12]:
BANNED_OUTPUT_IDS = [
    target_word_to_index.get(''),
    target_word_to_index.get('[UNK]'),
    target_word_to_index.get(START_TOKEN),
]
BANNED_OUTPUT_IDS = [idx for idx in BANNED_OUTPUT_IDS if idx is not None]

def translate_sentence(sentence, max_len=MAX_TARGET_LEN):
    sentence = clean_en(sentence)
    encoder_input = source_vectorizer(np.array([sentence])).numpy()
    decoded_words = [START_TOKEN]

    for _ in range(max_len - 1):
        decoder_text = ' '.join(decoded_words)
        decoder_input = target_vectorizer(np.array([decoder_text])).numpy()
        preds = model.predict([encoder_input, decoder_input], verbose=0)[0]
        token_scores = preds[len(decoded_words) - 1].copy()
        token_scores[BANNED_OUTPUT_IDS] = -1.0
        next_id = int(np.argmax(token_scores))
        next_word = target_index_lookup.get(next_id, '')

        if not next_word or next_word == END_TOKEN:
            break
        decoded_words.append(next_word)

    return ' '.join(w for w in decoded_words[1:] if w not in {START_TOKEN, END_TOKEN, '[UNK]'}).strip()

for text in [
    'a dog is running through a field',
    'a woman in a red shirt is standing in front of a building',
    'a man is climbing a rock wall',
]:
    print('EN:', text)
    print('AR:', translate_sentence(text))
    print()

EN: a dog is running through a field
AR: من الجيد أن تكون في غرفة لاحق

EN: a woman in a red shirt is standing in front of a building
AR: في الوقت الحالي ، في الحقيقة ، و من الأفضل أن تكون في البيت

EN: a man is climbing a rock wall
AR: - من فضلك لا يزال لديه القهوة



## Translate Malak's Generated Captions JSON

In [16]:
from pathlib import Path

INPUT_CAPTIONS_JSON = Path(r"C:\Users\polaa\Downloads\generated_captions.json")

OUTPUT_DIR = Path(r"C:\Users\polaa\.codex\worktrees\3b3b\Healix\outputs\translator")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(INPUT_CAPTIONS_JSON)
print("Exists:", INPUT_CAPTIONS_JSON.exists())
with open(INPUT_CAPTIONS_JSON, 'r', encoding='utf-8') as f:
    generated_captions = json.load(f)


C:\Users\polaa\Downloads\generated_captions.json
Exists: True


In [17]:
translated = {}

for i, (img_name, caption) in enumerate(generated_captions.items(), start=1):
    translated[img_name] = translate_sentence(caption)
    if i % 250 == 0:
        print(f'Translated {i}/{len(generated_captions)}')

output_json = OUTPUT_DIR / 'translated_captions2_ar.json'
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(translated, f, ensure_ascii=False, indent=2)

print('Saved:', output_json)
list(translated.items())[:5]

Translated 250/8091
Translated 500/8091
Translated 750/8091
Translated 1000/8091
Translated 1250/8091
Translated 1500/8091
Translated 1750/8091
Translated 2000/8091
Translated 2250/8091
Translated 2500/8091
Translated 2750/8091
Translated 3000/8091
Translated 3250/8091
Translated 3500/8091
Translated 3750/8091
Translated 4000/8091
Translated 4250/8091
Translated 4500/8091
Translated 4750/8091
Translated 5000/8091
Translated 5250/8091
Translated 5500/8091
Translated 5750/8091
Translated 6000/8091
Translated 6250/8091
Translated 6500/8091
Translated 6750/8091
Translated 7000/8091
Translated 7250/8091
Translated 7500/8091
Translated 7750/8091
Translated 8000/8091
Saved: C:\Users\polaa\.codex\worktrees\3b3b\Healix\outputs\translator\translated_captions2_ar.json


[('1000268201_693b08cb0e.jpg',
  'في الوقت الحالي ، في الحقيقة ، و من الأفضل أن تكون في البيت'),
 ('1001773457_577c3a7d70.jpg', 'من الجيد أن تكون في غرفة لاحق'),
 ('1002674143_1b742ab4b8.jpg',
  'من الجيد أن تكون في غرفة المدينة من أجل أن تكون في غرفة أنحاء الحياة'),
 ('1003163366_44323f5815.jpg',
  'في الوقت الحالي ، و من الأفضل أن تكون في المنزل ، و أنا سعيد'),
 ('1007129816_e794419615.jpg',
  'في الوقت الحالي ، في الحقيقة ، و من الأفضل أن تكون هنا')]